In [1]:
import numpy as np
import fastplotlib as fpl
import matplotlib.pyplot as plt
%matplotlib inline
import random
from tqdm.notebook import tqdm
import seaborn as sns
from qpsolvers import solve_qp
from itertools import product

wgpu version from git (0.15.3) and __version__ (0.19.3) don't match.
rendercanvas version from git (2.0.0) and __version__ (2.0.1) don't match.
WGPU: enumerate_adapters() is deprecated, use enumerate_adapters_sync() instead.
WGPU: request_adapter() is deprecated, use request_adapter_sync() instead.


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅,AMD Radeon 780M (RADV GFX1103_R1),IntegratedGPU,Vulkan,Mesa 24.2.8
✅ (default),NVIDIA RTX A1000 6GB Laptop GPU,DiscreteGPU,Vulkan,550.142
✅,NVIDIA RTX A1000 6GB Laptop GPU,DiscreteGPU,Vulkan,550.142
❗ limited,"llvmpipe (LLVM 19.1.0, 256 bits)",CPU,Vulkan,Mesa 24.2.8 (LLVM 19.1.0)
❌,"AMD Radeon 780M (radeonsi, gfx1103_r1, LLVM 19.1.0, DRM 3.59, 6.12.4-200.fc41.x86_64)",Unknown,OpenGL,4.6 (Core Profile) Mesa 24.2.8


Incompatible version of wgpu:
    Detected 0.15.3.post97+g8a0b267, need >=(0, 19, 0), <(0, 20, 0).
    To update, use e.g. `pip install -U wgpu`.


In [2]:
fpl.select_adapter(fpl.enumerate_adapters()[0])

In [4]:
def generate_data(
        n_timepoints: int,
        n_components: int,
        noise: float,
        n_pixels: int | tuple[int, int],  # n_pixels PER component, if a tuple a random number of pixles within the range is used
        n_pixels_bg,
        interp_factor: int = 2,
        firing_prop: float = 0.005,
) -> tuple[np.ndarray, np.ndarray]:
    """

    Parameters
    ----------
    n_timepoints: int

    n_components: int

    noise: float

    n_pixels: int, or (min, max) range
        n_pixels PER components, if (min, max) range is given then a random number of pixels within this range
        is used for each component

    n_pixels_bg: int

    interp_factor: int
        interpolation factor

    Returns
    -------
    np.ndarray, np.ndarray
        data [pixels, n_timepoints * interp_factor], truth labels: 1D array 0 is background

    """

    clean = np.zeros((n_components, n_timepoints * interp_factor))

    x = 0
    a = 0.7

    b = 0

    spikes = list()

    for i in range(n_components):
        spikes.append((np.random.rand(n_timepoints) < firing_prop).astype(float))

    for c_ix in range(n_components):
        for i in range(2, n_timepoints):
            clean[c_ix, i] = (a * clean[c_ix, i - 1]) + (b * clean[c_ix, i - 2]) + (1 * spikes[c_ix][i])

        x_ = np.arange(0, n_timepoints * interp_factor)
        xp_ = np.linspace(0, n_timepoints * interp_factor, n_timepoints)
        clean[c_ix] = np.interp(x_, xp_, fp=clean[c_ix, :n_timepoints])

    n_timepoints *= interp_factor

    series = list()
    labels = list()

    for i in range(n_components):
        # determine number of pixels for this component
        if isinstance(n_pixels, int):
            _n_pixels_iter = n_pixels
        elif isinstance(n_pixels, tuple):
            _n_pixels_iter = np.random.randint(low=n_pixels[0], high=n_pixels[1])

        for j in range(_n_pixels_iter):
            series.append(clean[i] + np.random.normal(scale=noise, size=n_timepoints))
            labels.append(i + 1)

    # series = np.vstack(series)

    bg_pixels = [np.random.normal(scale=noise, size=n_timepoints) for i in range(n_pixels_bg)]
    series = np.vstack([*series, *bg_pixels])
    labels = np.array([*labels, *[0 for i in range(len(bg_pixels))]])

    return series, labels

In [18]:
import torch

In [29]:
T_init = 100
noise = 1
k = n_components = 10
firing_prop = 0.05

In [22]:
X, _ = generate_data(
    n_timepoints=5_000,
    n_components=n_components,
    noise=noise,
    n_pixels=5,
    n_pixels_bg=0,
    interp_factor=1,
    firing_prop=firing_prop,
)

In [110]:
device = torch.device(0)
device

device(type='cuda', index=0)

In [96]:
device = "cpu"

In [111]:
X_init = torch.from_numpy(X[:, :T_init].astype(np.float32)).to(device)

In [112]:
error = 1e-3

In [113]:
# def sim_matching_init(X_init, k, error) -> tuple[torch.Tensor, torch.Tensor, int]:
#     """
#     Returns
#     -------
#         W, M, n_iter
#     """

cov = X_init.T @ X_init
XTX = cov / torch.linalg.norm(cov, ord="fro")

T = XTX.shape[0]

eta = 1e-1

Y = (torch.randn(k, T).to(device) / np.sqrt(T)).clip(0)

for n_iter in tqdm(range(25_000)):
    Y_old = Y
    Y = Y + eta * Y @ (XTX - Y.T @ Y)
    
    zeroed = Y.sum(axis=1) == 0.0
    Y[zeroed] = (torch.randn(zeroed.sum(), T).to(device) / np.sqrt(T)).clip(0)
    
    Y = Y.clip(0)

    er = torch.divide(torch.abs(Y - Y_old), torch.abs(Y_old + 1e-2)).max() / eta

    if er < error:
        break
        
W = Y @ X_init.T
M = Y @ Y.T

# return W, M, n_iter

  0%|          | 0/25000 [00:00<?, ?it/s]